# Checking for suitable data
This notebook is here to check whether suitable data is available for urban vegetation change in Singapore. 
- RQ: Where has the urban vegetation in-/decreased between year X and year Y, plus do the gains offset the losses?

1. First checking with openEO backend the CDSE data catalogue

In [1]:
import openeo
import rasterio
from localtileserver import TileClient, get_leaflet_tile_layer
import leafmap

In [2]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

1.1. Checking Sentinel2 Data

In [3]:
connection.describe_collection("SENTINEL2_L2A")

{'bands': [{'name': 'B01'},
  {'name': 'B02'},
  {'name': 'B03'},
  {'name': 'B04'},
  {'name': 'B05'},
  {'name': 'B06'},
  {'name': 'B07'},
  {'name': 'B08'},
  {'name': 'B8A'},
  {'name': 'B09'},
  {'name': 'B11'},
  {'name': 'B12'},
  {'name': 'WVP'},
  {'name': 'AOT'},
  {'name': 'SCL'},
  {'name': 'sunAzimuthAngles'},
  {'name': 'sunZenithAngles'},
  {'name': 'viewAzimuthMean'},
  {'name': 'viewZenithMean'},
  {'name': 'CLD'},
  {'name': 'SNW'}],
 'ceosard:specification': 'SR',
 'ceosard:specification_version': '5.0.1',
 'ceosard:type': 'optical',
 'cube:dimensions': {'bands': {'type': 'bands',
   'values': ['B01',
    'B02',
    'B03',
    'B04',
    'B05',
    'B06',
    'B07',
    'B08',
    'B8A',
    'B09',
    'B11',
    'B12',
    'WVP',
    'AOT',
    'SCL',
    'sunAzimuthAngles',
    'sunZenithAngles',
    'viewAzimuthMean',
    'viewZenithMean',
    'CLD',
    'SNW']},
  't': {'extent': ['2015-06-27T10:25:31Z', None], 'type': 'temporal'},
  'x': {'axis': 'x',
   'extent': [-180, 180],
   'reference_system': {'$schema': 'https://proj.org/schemas/v0.2/projjson.schema.json',
    'area': 'World',
    'bbox': {'east_longitude': 180,
     'north_latitude': 90,
     'south_latitude': -90,
     'west_longitude': -180},
    'coordinate_system': {'axis': [{'abbreviation': 'Lat',
       'direction': 'north',
       'name': 'Geodetic latitude',
       'unit': 'degree'},
      {'abbreviation': 'Lon',
       'direction': 'east',
       'name': 'Geodetic longitude',
       'unit': 'degree'}],
     'subtype': 'ellipsoidal'},
    'datum': {'ellipsoid': {'inverse_flattening': 298.257223563,
      'name': 'WGS 84',
      'semi_major_axis': 6378137},
     'name': 'World Geodetic System 1984',
     'type': 'GeodeticReferenceFrame'},
    'id': {'authority': 'OGC', 'code': 'Auto42001', 'version': '1.3'},
    'name': 'AUTO 42001 (Universal Transverse Mercator)',
    'type': 'GeodeticCRS'},
   'type': 'spatial'},
  'y': {'axis': 'y',
   'extent': [-90, 90],
   'reference_system': {'$schema': 'https://proj.org/schemas/v0.2/projjson.schema.json',
    'area': 'World',
    'bbox': {'east_longitude': 180,
     'north_latitude': 90,
     'south_latitude': -90,
     'west_longitude': -180},
    'coordinate_system': {'axis': [{'abbreviation': 'Lat',
       'direction': 'north',
       'name': 'Geodetic latitude',
       'unit': 'degree'},
      {'abbreviation': 'Lon',
       'direction': 'east',
       'name': 'Geodetic longitude',
       'unit': 'degree'}],
     'subtype': 'ellipsoidal'},
    'datum': {'ellipsoid': {'inverse_flattening': 298.257223563,
      'name': 'WGS 84',
      'semi_major_axis': 6378137},
     'name': 'World Geodetic System 1984',
     'type': 'GeodeticReferenceFrame'},
    'id': {'authority': 'OGC', 'code': 'Auto42001', 'version': '1.3'},
    'name': 'AUTO 42001 (Universal Transverse Mercator)',
    'type': 'GeodeticCRS'},
   'type': 'spatial'}},
 'description': 'SENTINEL-2 is a wide-swath, high-resolution, multi-spectral imaging mission, supporting Copernicus Land Monitoring studies, including the monitoring of vegetation, soil and water cover, as well as observation of inland waterways and coastal areas. The SENTINEL-2 Multispectral Instrument (MSI) samples 13 spectral bands: four bands at 10 metres, six bands at 20 metres and three bands at 60 metres spatial resolution.\n The level 2A data is atmospherically corrected using Sen2Cor.',
 'extent': {'spatial': {'bbox': [[-180, -90, 180, 90]]},
  'temporal': {'interval': [['2015-06-27T10:25:31Z', None]]}},
 'id': 'SENTINEL2_L2A',
 'keywords': ['COPERNICUS',
  'ESA',
  'Orthoimagery',
  'Sentinel-2',
  'MSI',
  'Level-2A',
  'Radiometry',
  'Plant Resource',
  'TOC'],
 'license': 'other',
 'links': [{'href': 'https://sentinels.copernicus.eu/documents/247904/690755/Sentinel_Data_Legal_Notice',
   'rel': 'license',
   'title': 'Legal notice on the use of Copernicus Sentinel Data and Service Information',
   'type': 'application/pdf'},
  {'href': 'https://stac.dataspace.cop

In [4]:
sentinel2_L2A_cube = connection.load_collection(
    "SENTINEL2_L2A",
    spatial_extent={"west": 103.60, "south": 1.20, "east": 104.05, "north": 1.48},
    temporal_extent=["2016-01-01", "2016-12-31"],
    bands=["B04", "B08", "SCL"], # red, nir, scene classification
    max_cloud_cover=70,)

In [ ]:
# Mask out cloud shadows, clouds and cirrus using the scene classification band
# SCL classification from https://custom-scripts.sentinel-hub.com/custom-scripts/sentinel-2/scene-classification/
scl = sentinel2_L2A_cube.band("SCL")
# 3 = cloud shadows, 8 = cloud medium probability, 9 = cloud high probability, 10 = thin cirrus
removed_scl_classes = (scl == 3) | (scl == 8) | (scl == 9) | (scl == 10)

sentinel2_L2A_cube = sentinel2_L2A_cube.mask(removed_scl_classes)

In [6]:
# median (due to potential outliers) composite over the year
sentinel2_L2A_composite = sentinel2_L2A_cube.reduce_dimension(dimension="t", reducer="median")

# Calculate NDVI
red = sentinel2_L2A_composite.band("B04")
nir = sentinel2_L2A_composite.band("B08")

ndvi = (nir - red) / (nir + red)

ndvi.download("../data/raw/raster/singapore_ndvi_s2_2016.tif")

KeyboardInterrupt: 

In [7]:
map = leafmap.Map(center=[1.35, 103.82], zoom=11)
map.add_raster("C:\\Users\\clair\\OneDrive\\Dokumente\\University Zurich\\Semester 7 (HS)\\sds320\\project\\data\\raw\\raster\\singapore_ndvi_s2_2016.tif", colormap="RdYlGn", vmin=-0.2, vmax=0.9)
map

Map(center=[1.340029, 103.8249635], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title'…

# 1.2. Checking Landsat Bi-monthly mosaic
Creating two NDVI composites to compare

In [3]:
import rasterio
import matplotlib.pyplot as plt
import openeo

In [4]:
connection = openeo.connect("openeo.dataspace.copernicus.eu")
connection.authenticate_oidc()

Authenticated using refresh token.


<Connection to 'https://openeo.dataspace.copernicus.eu/openeo/1.2/' with OidcBearerAuth>

In [5]:
connection.describe_collection("LANDSAT_BIMONTHLY_MOSAIC")

{'bands': [{'description': 'Blue', 'name': 'B01'},
  {'description': 'Green', 'name': 'B02'},
  {'description': 'Red', 'name': 'B03'},
  {'description': 'Near Infrared', 'name': 'B04'},
  {'description': 'Short-wave Infrared 1.6', 'name': 'B05'},
  {'description': 'Short-wave Infrared 2.2', 'name': 'B06'},
  {'description': 'Thermal', 'name': 'B07'}],
 'cube:dimensions': {'bands': {'type': 'bands',
   'values': ['B01', 'B02', 'B03', 'B04', 'B05', 'B06', 'B07']},
  't': {'extent': ['1997-01-01T00:00:00Z', '2024-12-31T00:00:00Z'],
   'type': 'temporal'},
  'x': {'axis': 'x',
   'extent': [-180, 180],
   'reference_system': 4326,
   'step': 0.00025,
   'type': 'spatial'},
  'y': {'axis': 'y',
   'extent': [-90, 90],
   'reference_system': 4326,
   'step': 0.00025,
   'type': 'spatial'}},
 'description': 'Global, cloud-free, and reconstructed historical Landsat spectral bands are provided at a spatial resolution of 30 meters and temporal resolution of bimonthly intervals from 1997 onwards. The dataset was generated using the GLAD Landsat ARD version 2 as the primary input for temporal aggregation and the imputation of missing values.',
 'experimental': True,
 'extent': {'spatial': {'bbox': [[-180, -90, 180, 90]]},
  'temporal': {'interval': [['1997-01-01T00:00:00Z',
     '2024-12-31T00:00:00Z']]}},
 'id': 'LANDSAT_BIMONTHLY_MOSAIC',
 'keywords': ['Landsat', 'Mosaic', 'Time-series', 'Global'],
 'license': 'other',
 'links': [{'href': 'https://www.usgs.gov/faqs/are-there-any-restrictions-use-or-redistribution-landsat-data',
   'rel': 'license',
   'title': 'Landsat data policy',
   'type': 'text/html'},
  {'href': 'https://stac.dataspace.copernicus.eu/v1/collections/opengeohub-landsat-bimonthly-mosaic-v1.0.1',
   'rel': 'derived_from',
   'type': 'application/json'},
  {'href': 'https://openeo.dataspace.copernicus.eu/openeo/1.2/collections',
   'rel': 'root'},
  {'href': 'https://openeo.dataspace.copernicus.eu/openeo/1.2/collections',
   'rel': 'parent'},
  {'href': 'https://openeo.dataspace.copernicus.eu/openeo/1.2/collections/LANDSAT_BIMONTHLY_MOSAIC',
   'rel': 'self'},
  {'href': 'https://openeo.dataspace.copernicus.eu/openeo/1.2/collections/LANDSAT_BIMONTHLY_MOSAIC/queryables',
   'rel': 'http://www.opengis.net/def/rel/ogc/1.0/queryables',
   'type': 'application/schema+json'}],
 'providers': [{'name': 'USGS',
   'roles': ['producer'],
   'url': 'https://www.usgs.gov/'},
  {'name': 'OpenGeoHub',
   'roles': ['processor'],
   'url': 'https://opengeohub.org/'},
  {'name': 'CloudFerro', 'roles': ['host'], 'url': 'https://cloudferro.com/'}],
 'sci:citation': 'OpenGeoHub [2025]',
 'sci:doi': '10.7717/peerj.18585',
 'stac_extensions': ['https://stac-extensions.github.io/alternate-assets/v1.2.0/schema.json',
  'https://stac-extensions.github.io/authentication/v1.1.0/schema.json',
  'https://stac-extensions.github.io/processing/v1.2.0/schema.json',
  'https://stac-extensions.github.io/projection/v2.0.0/schema.json',
  'https://stac-extensions.github.io/scientific/v1.0.0/schema.json',
  'https://stac-extensions.github.io/storage/v2.0.0/schema.json'],
 'stac_version': '1.1.0',
 'summaries': {'bands': [{'description': 'Blue',
    'eo:center_wavelength': 0.482,
    'eo:common_name': 'blue',
    'eo:full_width_half_max': 0.06,
    'name': 'B01'},
   {'description': 'Green',
    'eo:center_wavelength': 0.561,
    'eo:common_name': 'green',
    'eo:full_width_half_max': 0.057,
    'name': 'B02'},
   {'description': 'Red',
    'eo:center_wavelength': 0.655,
    'eo:common_name': 'red',
    'eo:full_width_half_max': 0.037,
    'name': 'B03'},
   {'description': 'Near Infrared',
    'eo:center_wavelength': 0.865,
    'eo:common_name': 'nir08',
    'eo:full_width_half_max': 0.028,
    'name': 'B04'},
   {'description': 'Short-wave Infrared 1.6',
    'eo:center_wavelength': 1.609,
    'eo:common_name': 'swir16',
    'eo:full_width_half_max': 0.085,
    'name': 'B05'},
   {'description': 'Short-wave Infrared 2.2',
    'eo:center_wavelength': 2.201,

In [6]:
# Bounding box covering the whole of Singapore
singapore = {"west": 103.60, "south": 1.20, "east": 104.05, "north": 1.48}

Sing_landsat_2024_cube = connection.load_collection(
    "LANDSAT_BIMONTHLY_MOSAIC",
    spatial_extent=singapore,
    temporal_extent=["2024-05-01", "2024-06-30"],
    bands=["B03", "B04"],)

In [7]:
red = Sing_landsat_2024_cube.band("B03")
nir = Sing_landsat_2024_cube.band("B04")

ndvi = (nir - red) / (nir + red)

ndvi.download("../data/processed/singapore_ndvi_landsat_2024.tif")

In [ ]:
# receiving error codes when using matplotlib to visualize the raster
with rasterio.open("../data/processed/singapore_ndvi_landsat_2024.tif") as src:
    data = src.read(1)

plt.figure(figsize=(12, 6))
plt.imshow(data, cmap="RdYlGn", vmin=-0.2, vmax=0.9)
plt.colorbar(label="NDVI")
plt.title("Singapore NDVI (May to June 2024)")
plt.show()

: 

In [3]:
#using leafmap to visualise the raster

client = TileClient("../data/processed/singapore_ndvi_landsat_2024.tif", host="127.0.0.1")
layer = get_leaflet_tile_layer(client, cmap="rdylgn", vmin=-0.2, vmax=0.9)

m = leafmap.Map(center=[1.35, 103.82], zoom=11)
m.add_layer(layer)
m

Map(center=[1.35, 103.82], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_o…

In [4]:
# Bounding box covering the whole of Singapore
singapore = {"west": 103.60, "south": 1.20, "east": 104.05, "north": 1.48}

Sing_landsat_1999_cube = connection.load_collection(
    "LANDSAT_BIMONTHLY_MOSAIC",
    spatial_extent=singapore,
    temporal_extent=["1999-05-01", "1999-06-30"],
    bands=["B03", "B04"],)

In [5]:
red = Sing_landsat_1999_cube.band("B03")
nir = Sing_landsat_1999_cube.band("B04")

ndvi = (nir - red) / (nir + red)

ndvi.download("../data/processed/singapore_ndvi_landsat_1999.tif")

In [6]:
#using leafmap to visualise the raster

client = TileClient("../data/processed/singapore_ndvi_landsat_1999.tif", host="127.0.0.1")
layer = get_leaflet_tile_layer(client, cmap="rdylgn", vmin=-0.2, vmax=0.9)

m = leafmap.Map(center=[1.35, 103.82], zoom=11)
m.add_layer(layer)
m

Map(center=[1.35, 103.82], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_o…

# Checking the singapore governmental data access (vector)

link to the government data https://data.gov.sg/datasets?query=Master+Plan+planning+area+boundaries&resultId=d_4765db0e87b9c86336792efe8a1f7a66

In [7]:
import geopandas as gpd

In [10]:

singapore_2019_planning_areas = gpd.read_file("../data/raw/vectors/MasterPlan2019PlanningAreaBoundaryNoSea.geojson")
print(singapore_2019_planning_areas.crs)
print(len(singapore_2019_planning_areas))
singapore_2019_planning_areas.head()

EPSG:4326
55


,OBJECTID,PLN_AREA_N,PLN_AREA_C,CA_IND,REGION_N,REGION_C,INC_CRC,FMEL_UPD_D,SHAPE.AREA,SHAPE.LEN,geometry
0,1,BEDOK,BD,N,EAST REGION,ER,5F00E6FF084F3364,20191223152014,2.173397e+07,21864.234336,"POLYGON ((103.93208 1.30555, 103.93208 1.30555..."
1,2,BOON LAY,BL,N,WEST REGION,WR,C96AED188C00B2FC,20191223152014,8.282808e+06,18542.688792,"POLYGON ((103.72042 1.32824, 103.72066 1.32865..."
2,3,BUKIT BATOK,BK,N,WEST REGION,WR,3BEC4C829160F28A,20191223152014,1.114018e+07,15236.442232,"POLYGON ((103.76408 1.37001, 103.76385 1.37036..."
3,4,BUKIT MERAH,BM,N,CENTRAL REGION,CR,4850795BB0B6A4F7,20191223152014,1.446120e+07,29161.409834,"POLYGON ((103.82361 1.26018, 103.82368 1.26018..."
4,5,BUKIT PANJANG,BP,N,WEST REGION,WR,656F87D23D6DAB02,20191223152014,9.019940e+06,15891.853279,"POLYGON ((103.77445 1.39029, 103.77444 1.39029..."


In [11]:
m = leafmap.Map(center=[1.35, 103.82], zoom=11)
m.add_gdf(singapore_2019_planning_areas, layer_name="Planning areas")
m

Map(center=[1.35, 103.82], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_o…